In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from hmmlearn import hmm
from hmmlearn.hmm import GaussianHMM
import joblib
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
import matplotlib.dates as mdates
from matplotlib.lines import Line2D

## TRAINING GAUSSIAN HMM

In [2]:
# Đọc dữ liệu
train_features=pd.read_csv('traffic_train_with_features.csv')

In [3]:
# Training chính thức model
# =========================
# 1. FEATURE SET CHÍNH THỨC + N
# =========================
best_features = ['traffic_volume', 'avg_24h', 'volume_deviation']
N = 4

# =========================
# 2. CHUẨN BỊ DỮ LIỆU
# =========================
X_train = train_features[best_features].dropna().values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# =========================
# 3. TRAIN FINAL HMM MODEL
# =========================
model = GaussianHMM(
    n_components=N,
    covariance_type="full",
    n_iter=200,
    random_state=42
)

model.fit(X_train_scaled)


# =========================
#  LƯU MODEL + SCALER
# =========================
joblib.dump(model, "gaussian_hmm_model.pkl")
joblib.dump(scaler, "gaussian_hmm_scaler.pkl")

print("\n✔ Model và scaler đã được lưu thành công!")


✔ Model và scaler đã được lưu thành công!


## Kết quả Training Gaussian 

In [ ]:
# --- Lấy kết quả suy diễn từ Gaussian ---
final_n=4

# Dự đoán state và xác suất trên tập train
train_states = model.predict(X_train_scaled)
train_probs = model.predict_proba(X_train_scaled)
train_max_probs = train_probs.max(axis=1)

# Gắn output vào dataframe train
train_features['gaussian_hmm_state'] = train_states
train_features['gaussian_hmm_prob'] = train_max_probs

# Lưu file train có output HMM
train_features.to_csv("traffic_train_hmm_output.csv", index=False)

# --- TRÍCH SUẤT OUTPUT ---


# Xác suất khởi đầu
startprob_df = pd.DataFrame({
    "State": [f"State_{i}" for i in range(final_n)],
    "Start Probability": model.startprob_.round(6)
})

print("START PROBABILITIES - FINAL GAUSSIAN HMM")
display(startprob_df)

# 1. Ma trận chuyển trạng thái
transition_df = pd.DataFrame(
    model.transmat_.round(4),
    columns=[f"To_State_{i}" for i in range(final_n)],
    index=[f"From_State_{i}" for i in range(final_n)]
)

print("TRANSITION MATRIX - FINAL GAUSSIAN HMM")
display(transition_df)

# 2. Trung bình phát xạ của từng state
means_df = pd.DataFrame(
    model.means_.round(4),
    columns=best_features,
    index=[f"State_{i}" for i in range(final_n)]
)

print("STATE MEANS - FINAL GAUSSIAN HMM")
display(means_df)

# 3. Phân bố state
state_dist_df = (
    train_features['gaussian_hmm_state']
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
    .reset_index()
)

state_dist_df.columns = ["State", "Percentage (%)"]

print("STATE DISTRIBUTION - FINAL GAUSSIAN HMM")
display(state_dist_df)

# 4. Tóm tắt xác suất suy diễn
prob_summary_df = (
    train_features
    .groupby('gaussian_hmm_state')['gaussian_hmm_prob']
    .agg(['mean', 'min', 'max'])
    .round(4)
    .reset_index()
)

prob_summary_df.columns = [
    "State",
    "Mean Probability",
    "Min Probability",
    "Max Probability"
]

print("PROBABILITY SUMMARY - FINAL GAUSSIAN HMM")
display(prob_summary_df)



START PROBABILITIES - FINAL GAUSSIAN HMM


,State,Start Probability
0,State_0,0.0
1,State_1,1.0
2,State_2,0.0
3,State_3,0.0


TRANSITION MATRIX - FINAL GAUSSIAN HMM


,To_State_0,To_State_1,To_State_2,To_State_3
From_State_0,0.9191,0.0000,0.0000,0.0809
From_State_1,0.0000,0.9137,0.0836,0.0027
From_State_2,0.0028,0.0705,0.9118,0.0149
From_State_3,0.0661,0.0269,0.0038,0.9031


STATE MEANS - FINAL GAUSSIAN HMM


,traffic_volume,avg_24h,volume_deviation
State_0,-0.9574,-1.4240,-0.6081
State_1,1.0346,0.6276,0.8927
State_2,-0.8435,0.6730,-1.0340
State_3,0.5026,-0.7444,0.7049


STATE DISTRIBUTION - FINAL GAUSSIAN HMM


,State,Percentage (%)
0,0,17.57
1,1,31.33
2,2,30.81
3,3,20.29


PROBABILITY SUMMARY - FINAL GAUSSIAN HMM


,State,Mean Probability,Min Probability,Max Probability
0,0,0.9606,0.3948,1.0
1,1,0.9810,0.4521,1.0
2,2,0.9807,0.3720,1.0
3,3,0.9470,0.3746,1.0


## Check Gaussian emission sau Trainings

In [10]:
from scipy.stats import skew, kurtosis, normaltest
# Check Gaussian hmm cho từng nhóm state
train_feature1= pd.read_csv('traffic_train_hmm_output.csv')
df1 = train_feature1.copy()

best_features = ['traffic_volume', 'avg_24h', 'volume_deviation']
feature_cols = best_features  # hoặc list feature bạn chọn


# =========================
# Check Gaussian theo state có sẵn
# =========================
for state_id in sorted(df["gaussian_hmm_state"].unique()):

    print(f"\n================ STATE {state_id} ================")

    subset = df1[df1["gaussian_hmm_state"] == state_id]

    for col in feature_cols:
        data = subset[col].values

        print(f"\nFeature: {col}")

        print(f"Mean: {np.mean(data):.4f}")
        print(f"Std : {np.std(data):.4f}")
        print(f"Skew: {skew(data):.4f}")
        print(f"Kurt: {kurtosis(data):.4f}")

        stat, p = normaltest(data)


================ STATE 0 ================

Feature: traffic_volume
Mean: 1410.2078
Std : 1012.8817
Skew: 0.7440
Kurt: -0.2758

Feature: avg_24h
Mean: 2584.2052
Std : 435.2737
Skew: -1.2800
Kurt: 4.5725

Feature: volume_deviation
Mean: -1173.9974
Std : 1175.6454
Skew: 0.8229
Kurt: 0.1078

================ STATE 1 ================

Feature: traffic_volume
Mean: 5330.5319
Std : 689.1615
Skew: 0.2566
Kurt: -0.6808

Feature: avg_24h
Mean: 3605.0097
Std : 174.9923
Skew: -0.0148
Kurt: -0.7749

Feature: volume_deviation
Mean: 1725.5222
Std : 661.6472
Skew: 0.3216
Kurt: -0.7150

================ STATE 2 ================

Feature: traffic_volume
Mean: 1647.5848
Std : 1132.3185
Skew: 0.3332
Kurt: -1.3853

Feature: avg_24h
Mean: 3627.9637
Std : 190.6545
Skew: -0.0323
Kurt: -0.6167

Feature: volume_deviation
Mean: -1980.3789
Std : 1147.4019
Skew: 0.2636
Kurt: -1.4283

================ STATE 3 ================

Feature: traffic_volume
Mean: 4286.7362
Std : 827.9061
Skew: 0.2575
Kurt: 0.0257

Featur